In [1]:
from collections import defaultdict

import pandas as pd
import os
import json

from nlp_content_validity.data.read_data import read_dataset, read_validity

dataset='colquitt_et_al'
# val_metric='htd'
# nlp_metric='htd_nlp'
val_metric='htc'
nlp_metric='mean_focal'
basedir = f'../../data/interim/{dataset}'
models = list(map(lambda x: os.path.splitext(x)[0], filter(lambda x: x.endswith('json'), os.listdir(basedir))))
model = 'sentence_t5'
# for model in models:
# for val_metric, nlp_metric in [('htd', 'htd_nlp'), ('htc', 'mean_focal')]:
definitions, df_, focal_scales, orbiting_dict = read_dataset(dataset)
relations = {(focal, scale2): label for focal, vals in orbiting_dict.items() for label, scale2 in vals.items()}
for focal in focal_scales:
    relations[(focal, focal)] = 'focal'
filename = f'{model}.json'
model_name = os.path.splitext(filename)[0]
# print(f'processing {basedir}/{filename}')
with open(f'{basedir}/{filename}', 'rb') as f:
    data = json.load(f)
data_dict = defaultdict(dict)
for i in data:
    for k, v in i.items():
        for kk, vv in v.items():
            data_dict[k][relations[(k, kk)]] = vv

similarities=list()
for scale, vals in data_dict.items():
    df=pd.DataFrame(vals)
    df['scale']=scale
    df['item'] = df_[df_.scale==scale].item.values
    similarities.append(df)
df = pd.concat(similarities)
if val_metric == 'htd':
    df[f'item_{nlp_metric}'] = (2*df.focal)-df.orbiting_scale_1-df.orbiting_scale_2
elif val_metric == 'htc':
    df[f'item_{nlp_metric}'] = df.focal
else:
    raise NotImplementedError(f'metric {val_metric} is not implemented')
validities = read_validity(dataset)
df = pd.merge(df, validities, left_on='scale', right_on='focal_scale', how='left')

nlp_metrics = pd.read_csv(f'../../data/processed/{dataset}/{model}.csv', index_col=0).rename(columns={'htd':"htd_nlp"})
merged = pd.merge(validities, nlp_metrics, left_index=True, right_index=True)

merged[f'{val_metric}_rank_diff'] = (merged[f'{val_metric}_norm'].rank()-merged[f'{nlp_metric}_norm'].rank())

df['item_rank_diff'] = df[f'{val_metric}'].rank() - df[f'item_{nlp_metric}'].rank()


In [2]:
import spacy
nlp = spacy.load('en_core_web_trf')

In [3]:
import textacy

In [34]:
import numpy as np


In [39]:
def featurize(txt):
    doc = nlp(txt)
    # - N. sentences, words, chars
    n_sents =textacy.text_stats.n_sents(doc)
    n_words =textacy.text_stats.n_words(doc)
    n_chars =textacy.text_stats.n_chars(doc)
    # - Readability index (ARI, smog, coleman liau, flesch kincaid, gunning fog)
    smog_index = textacy.text_stats.readability.smog_index(doc)
    # - Freq. of parts-of-speech
    pos = {k:v/len(doc) for k, v in textacy.text_stats.counts.pos(doc).items()}
    # - Frac. of function words, e.g. for, to, the.
    stopwords = sum(tok.is_stop for tok in doc)/len(doc)
    # - Median chars/syllables per word
    n_chars_per_word = np.mean(textacy.text_stats.basics.n_chars_per_word(doc))
    n_syllables_per_word = np.mean(textacy.text_stats.basics.n_syllables_per_word(doc))
    # - Frac. of punctuation (e.g. ‘,’ ‘.’)
    punct=sum(tok.is_punct for tok in doc)/len(doc)
    # - Type-token ratio
    log_ttr =textacy.text_stats.diversity.log_ttr(doc)
    return dict(smog_index=smog_index,
        stopwords=stopwords,
        n_chars_per_word=n_chars_per_word,
        n_syllables_per_word=n_syllables_per_word,
        punct=punct,
        n_sents=n_sents,
        n_words=n_words,
        n_chars=n_chars,
        log_ttr=log_ttr) |pos

In [85]:
defs =pd.Series(definitions)
defs_featurized = pd.DataFrame(defs.apply(featurize).tolist(), index=defs.index).fillna(0.).rename(columns=lambda x: 'definition_'+x)

In [75]:
scale_items=df_.groupby('scale').item.agg(lambda x:'\n '.join(x))

In [76]:
featurized =pd.DataFrame(scale_items.apply(featurize).tolist(), index=scale_items.index).fillna(0.)

In [77]:
n_items =df_.groupby('scale').size()

In [78]:
featurized['n']=n_items

In [87]:
featurized = featurized.rename(columns=lambda x: 'item_'+x)

In [89]:
featurized =pd.merge(defs_featurized, featurized, left_index=True, right_index=True)

In [97]:
featurized =pd.merge(featurized, merged[[f'{val_metric}_rank_diff']], left_index=True, right_index=True)

In [98]:
X = featurized[[i for i in featurized.columns if i != f'{val_metric}_rank_diff']]
y=featurized[f'{val_metric}_rank_diff']

In [100]:
y

Avoidant Leader Behaviors    36.0
Bottom-line mentality        -1.0
Calling                      11.5
Career Satisfaction           5.0
Citizenship Fatigue         -33.5
                             ... 
Tendency to Gossip           23.5
Victim Identity               2.5
Volunteering                 23.0
Voracity                      4.5
Wanderlust                  -37.5
Name: htc_rank_diff, Length: 112, dtype: float64